# Section 5: Results and Economic Interpretation

This section presents the final Model 3 estimates, their economic meaning and uncertainty, and the model’s explanatory and predictive performance.


<a id="section-5a"></a>

## 5(a) Professional Final Regression Table


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display, Markdown

%matplotlib inline

root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
             if (folder / "data/processed/parking_cross_section.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("The project data/processed/parking_cross_section.csv file is required.")
parking_full = pd.read_csv(root / "data/processed/parking_cross_section.csv",
                          dtype={"SpaceID": str, "GEOID": str})
parking = parking_full.loc[parking_full["RateType"].isin(["FLAT", "TOD"])].copy()
dependent_variable = "occupancy_rate"
candidate_variables = [
    "price_midpoint", "time_limit_hours", "population_density", "median_income",
    "nearby_restaurant_retail_count", "drive_alone_share", "RateType", "MeterType",
    "housing_density", "transit_share", "walk_share", "nearby_business_count"
]
encoded = pd.get_dummies(parking[candidate_variables], drop_first=True, dtype=int)
data = pd.concat([parking[["occupancy_rate"]], encoded], axis=1).dropna().copy()
y = data["occupancy_rate"]
X = data.drop(columns="occupancy_rate")
assert parking_full["SpaceID"].is_unique
assert parking.loc[data.index, "SpaceID"].is_unique
print("Original / FLAT+TOD / common:", len(parking_full), len(parking), len(data))
print("Candidates / common-sample tracts:", X.shape[1], parking.loc[data.index, "GEOID"].nunique())
# Fixed rule: retain EVERY slope with conventional p < 0.05 in the
# original 12-candidate full OLS model, on the same complete-case sample.
# This is one screen, not repeated p-value deletion or hold-out tuning.
full_model = sm.OLS(y, sm.add_constant(X)).fit()
alpha = 0.05
predictors = [v for v in X.columns if full_model.pvalues[v] < alpha]
legacy_predictors = ["price_midpoint", "time_limit_hours", "population_density",
                     "nearby_restaurant_retail_count", "drive_alone_share",
                     "transit_share", "RateType_TOD"]
X1 = sm.add_constant(data[["price_midpoint"]])
X2 = sm.add_constant(data[["price_midpoint", "population_density", "nearby_restaurant_retail_count"]])
X3 = sm.add_constant(data[predictors])
model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()
hc1 = sm.OLS(y, X3).fit(cov_type="HC1")
legacy_model = sm.OLS(y, sm.add_constant(data[legacy_predictors])).fit()
assert set(predictors) == set(full_model.pvalues.drop("const").loc[lambda p: p < alpha].index)

# Preserve complete table text in report outputs.
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)


Original / FLAT+TOD / common: 4139 4060 4016
Candidates / common-sample tracts: 12 45


In [2]:
regression_table=pd.DataFrame({"OLS coefficient":model3.params,"Conventional SE":model3.bse,
 "HC1 SE":hc1.bse,"Conventional p":model3.pvalues,"HC1 p":hc1.pvalues,
 "HC1 95% lower":hc1.conf_int()[0],"HC1 95% upper":hc1.conf_int()[1]})
regression_table.index.name="Parameter"
display(regression_table.style.format("{:.6g}"))
display(Markdown(f"**Outcome:** monthly occupancy fraction. **N:** {int(model3.nobs):,}; **slopes:** {len(predictors)}; **R²:** {model3.rsquared:.5f}; **adjusted R²:** {model3.rsquared_adj:.5f}. HC1 uses statsmodels' default normal-reference inference; conventional and robust intervals should not be mixed."))


,OLS coefficient,Conventional SE,HC1 SE,Conventional p,HC1 p,HC1 95% lower,HC1 95% upper
Parameter,,,,,,,
const,0.319517,0.00938566,0.0105497,2.35919e-223,1.70823e-201,0.29884,0.340194
price_midpoint,0.0515432,0.00231509,0.00224949,1.24623e-103,3.42795e-116,0.0471343,0.0559521
population_density,3.96115e-06,2.16052e-07,2.27304e-07,3.64525e-72,5.18242e-68,3.51564e-06,4.40665e-06
nearby_restaurant_retail_count,-0.000534366,4.21681e-05,3.8711e-05,4.11596e-36,2.41157e-43,-0.000610238,-0.000458494
transit_share,0.100461,0.0449134,0.0459653,0.0253558,0.0288457,0.0103709,0.190552
walk_share,0.282333,0.0286379,0.0267372,1.13582e-22,4.58909e-26,0.229929,0.334737
nearby_business_count,6.66312e-05,9.27183e-06,8.88582e-06,7.89437e-13,6.45059e-14,4.92153e-05,8.40471e-05
RateType_TOD,0.0435671,0.00705204,0.00809947,7.14119e-10,7.4896e-08,0.0276925,0.0594418


**Outcome:** monthly occupancy fraction. **N:** 4,016; **slopes:** 7; **R²:** 0.29414; **adjusted R²:** 0.29291. HC1 uses statsmodels' default normal-reference inference; conventional and robust intervals should not be mixed.

<a id="section-5b"></a>

## 5(b) Economic Interpretation of All Coefficients


In [3]:
changes=pd.Series({"price_midpoint":1,"population_density":1000,
 "nearby_restaurant_retail_count":100,"transit_share":.1,"walk_share":.1,
 "nearby_business_count":100,"RateType_TOD":1})
labels={"price_midpoint":"$1 higher stated hourly proxy",
 "population_density":"1,000 more people per sq mile",
 "nearby_restaurant_retail_count":"100 more food/retail registrations, total fixed",
 "transit_share":"10-pp higher resident transit share",
 "walk_share":"10-pp higher resident walk share",
 "nearby_business_count":"100 more total registrations, food/retail fixed",
 "RateType_TOD":"TOD versus FLAT"}
effects=pd.DataFrame({"Comparison":[labels[v] for v in predictors],
 "Occupancy difference (pp)":100*hc1.params[predictors]*changes[predictors],
 "HC1 lower (pp)":100*hc1.conf_int().loc[predictors,0]*changes[predictors],
 "HC1 upper (pp)":100*hc1.conf_int().loc[predictors,1]*changes[predictors],
 "HC1 p":hc1.pvalues[predictors]})
display(effects.style.format({c:"{:.4g}" for c in effects if c!="Comparison"}))
for v in predictors:
    effect=effects.loc[v,"Occupancy difference (pp)"]
    lo,hi=effects.loc[v,["HC1 lower (pp)","HC1 upper (pp)"]]
    display(Markdown(f"- **{labels[v]}:** associated with **{abs(effect):.3f} percentage points {'higher' if effect>=0 else 'lower'} occupancy**, holding other included covariates fixed; HC1 95% CI **[{lo:.3f}, {hi:.3f}] pp**."))
display(Markdown(f"**Intercept:** {model3.params['const']:.5f} occupancy, or {100*model3.params['const']:.2f}%, for a FLAT space with zero values for all numeric covariates. This joint zero-covariate profile is outside observed support, so it is an algebraic baseline, not an estimate for a typical parking space."))


,Comparison,Occupancy difference (pp),HC1 lower (pp),HC1 upper (pp),HC1 p
price_midpoint,$1 higher stated hourly proxy,5.154,4.713,5.595,3.428e-116
population_density,"1,000 more people per sq mile",0.3961,0.3516,0.4407,5.182e-68
nearby_restaurant_retail_count,"100 more food/retail registrations, total fixed",-5.344,-6.102,-4.585,2.412e-43
transit_share,10-pp higher resident transit share,1.005,0.1037,1.906,0.02885
walk_share,10-pp higher resident walk share,2.823,2.299,3.347,4.589e-26
nearby_business_count,"100 more total registrations, food/retail fixed",0.6663,0.4922,0.8405,6.451e-14
RateType_TOD,TOD versus FLAT,4.357,2.769,5.944,7.49e-08


- **$1 higher stated hourly proxy:** associated with **5.154 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[4.713, 5.595] pp**.

- **1,000 more people per sq mile:** associated with **0.396 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[0.352, 0.441] pp**.

- **100 more food/retail registrations, total fixed:** associated with **5.344 percentage points lower occupancy**, holding other included covariates fixed; HC1 95% CI **[-6.102, -4.585] pp**.

- **10-pp higher resident transit share:** associated with **1.005 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[0.104, 1.906] pp**.

- **10-pp higher resident walk share:** associated with **2.823 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[2.299, 3.347] pp**.

- **100 more total registrations, food/retail fixed:** associated with **0.666 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[0.492, 0.840] pp**.

- **TOD versus FLAT:** associated with **4.357 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[2.769, 5.944] pp**.

**Intercept:** 0.31952 occupancy, or 31.95%, for a FLAT space with zero values for all numeric covariates. This joint zero-covariate profile is outside observed support, so it is an algebraic baseline, not an estimate for a typical parking space.

Both business counts are in this specification. Holding the total fixed, a higher food/retail count represents a change in registration composition; holding food/retail fixed, a higher total represents other registrations. The negative conditional food/retail coefficient is therefore not the effect of opening new businesses. Shares describe tract residents, and a 10-percentage-point share change is 0.10 in fraction units. No coefficient is interpreted as a proven causal effect.

<a id="section-5c"></a>

## 5(c) Statistical vs. Economic Significance


In [4]:
display(pd.DataFrame({"HC1 p":hc1.pvalues[predictors],"Significant at 5%":hc1.pvalues[predictors]<.05,
                      "Scaled occupancy pp":effects["Occupancy difference (pp)"]}))
display(Markdown(f"A $1 price comparison is {effects.loc['price_midpoint','Occupancy difference (pp)']:.2f} occupancy pp, against a common-sample mean of {100*y.mean():.2f}%. Economic relevance depends on the realistic price change and local support, not just the p-value. Statistical significance is conditional on selection and the covariance assumptions and does not establish causality."))


,HC1 p,Significant at 5%,Scaled occupancy pp
price_midpoint,3.427946e-116,True,5.154319
population_density,5.182421e-68,True,0.396115
nearby_restaurant_retail_count,2.411570e-43,True,-5.343657
transit_share,2.884573e-02,True,1.004612
walk_share,4.589094e-26,True,2.823330
nearby_business_count,6.450594e-14,True,0.666312
RateType_TOD,7.489605e-08,True,4.356712


A $1 price comparison is 5.15 occupancy pp, against a common-sample mean of 53.76%. Economic relevance depends on the realistic price change and local support, not just the p-value. Statistical significance is conditional on selection and the covariance assumptions and does not establish causality.

<a id="section-5d"></a>

## 5(d) Confidence Intervals for Key Parameters


The tables in 5(a)–5(b) calculate HC1 95% intervals for every parameter and express every slope interval in the same scaled economic units. An interval spanning zero does not establish a nonzero association; an interval excluding zero does not establish an exogenous price effect. The 95% interpretation refers to long-run coverage under the fitted inference assumptions, not a 95% probability attached to a fixed unknown coefficient. Prior selection and correlated neighborhood errors can invalidate nominal coverage; Section 4(e) adds conditional row-bootstrap intervals without claiming to solve those issues.

In [5]:
price_interval_pp = 100 * hc1.conf_int().loc["price_midpoint"]
display(Markdown(f"For a **one-dollar higher stated price proxy**, the HC1 95% interval is **{price_interval_pp.iloc[0]:.3f} to {price_interval_pp.iloc[1]:.3f} occupancy percentage points**, holding included covariates fixed. It excludes zero under the fitted inference assumptions, supporting a positive conditional association; it does not identify the occupancy effect of an imposed rate increase. The density, registration, share and TOD intervals in 5(b) use their corresponding economic comparison units, so precision is assessed on the same scale as the estimated change."))


For a **one-dollar higher stated price proxy**, the HC1 95% interval is **4.713 to 5.595 occupancy percentage points**, holding included covariates fixed. It excludes zero under the fitted inference assumptions, supporting a positive conditional association; it does not identify the occupancy effect of an imposed rate increase. The density, registration, share and TOD intervals in 5(b) use their corresponding economic comparison units, so precision is assessed on the same scale as the estimated change.

<a id="section-5e"></a>

## 5(e) R² and Model Fit in Economic Context


In [6]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
folds = KFold(n_splits=5, shuffle=True, random_state=10)
fits = {"Model 1": model1, "Model 2": model2, "Model 3": model3,
        "Seven-predictor sensitivity": legacy_model}
fold_errors = {}
for label, fit in fits.items():
    columns = [v for v in fit.model.exog_names if v != "const"]
    fold_errors[label] = -cross_val_score(LinearRegression(), data[columns], y,
                                        cv=folds, scoring="neg_root_mean_squared_error")
cv_results = pd.DataFrame(fold_errors, index=pd.Index(range(1, 6), name="Held-out fold"))
comparison = pd.DataFrame({
    label: {"Slopes": len(fit.params)-1, "N": int(fit.nobs), "R2": fit.rsquared,
            "Adjusted R2": fit.rsquared_adj, "AIC": fit.aic, "BIC": fit.bic,
            "In-sample RMSE": np.sqrt(np.mean(fit.resid**2)),
            "Mean 5-fold RMSE": fold_errors[label].mean(),
            "SD across folds": fold_errors[label].std(ddof=1)}
    for label, fit in fits.items()
}).T
display(cv_results.round(6))
display(comparison.round(6))
display(Markdown(f"Model 3 explains **{100*model3.rsquared:.2f}%** of sample occupancy variation; **{100*(1-model3.rsquared):.2f}%** remains unexplained. Its mean five-fold RMSE is **{100*fold_errors['Model 3'].mean():.2f} occupancy pp**. This monthly cross-sectional error is not an instantaneous availability probability or out-of-city validation."))


,Model 1,Model 2,Model 3,Seven-predictor sensitivity
Held-out fold,,,,
1,0.175393,0.165986,0.162474,0.162926
2,0.179741,0.170995,0.163980,0.166667
3,0.179111,0.170807,0.163985,0.167118
4,0.176845,0.165379,0.162830,0.164252
5,0.176728,0.167872,0.160709,0.165612


,Slopes,N,R2,Adjusted R2,AIC,BIC,In-sample RMSE,Mean 5-fold RMSE,SD across folds
Model 1,1.0,4016.0,0.156775,0.156565,-2483.670513,-2471.074430,0.177522,0.177564,0.001807
Model 2,3.0,4016.0,0.243498,0.242932,-2915.517259,-2890.325092,0.168146,0.168208,0.002626
Model 3,7.0,4016.0,0.294138,0.292905,-3185.769994,-3135.385661,0.162421,0.162796,0.001349
Seven-predictor sensitivity,7.0,4016.0,0.273623,0.272354,-3070.712410,-3020.328077,0.164764,0.165315,0.001732


Model 3 explains **29.41%** of sample occupancy variation; **70.59%** remains unexplained. Its mean five-fold RMSE is **16.28 occupancy pp**. This monthly cross-sectional error is not an instantaneous availability probability or out-of-city validation.

The unexplained variation leaves substantial differences in occupancy across otherwise similar spaces. Unobserved parking supply, enforcement, destinations and time-specific demand may contribute. A higher R² does not establish exogeneity or reliable transfer to other neighborhoods; the held-out error is a within-setting prediction measure, and random folds can share tract conditions.

### Comparison with academic literature

Shoup’s (2006) search-cost mechanism concerns the curb/off-street price tradeoff and scarce spaces. Our positive conditional price association need not contradict it: a cross-location regression can combine strong demand with officials’ higher tariffs, and our outcome does not measure search time. [Shoup (2006)](https://escholarship.org/content/qt55s7079f/qt55s7079f.pdf).

Pierce and Shoup (2013) report an average elasticity about −0.4 from SFpark adjustments, whereas our level coefficient is an occupancy-fraction difference per dollar across LA spaces. The units and designs differ. A positive LA association cannot be interpreted as an opposite causal elasticity; source dates, omitted demand and tariff assignment differ. [Pierce and Shoup (2013)](https://www.its.ucla.edu/wp-content/uploads/sites/10/2017/06/Getting-the-Prices-Right-An-Evaluation-of-Pricing-Parking-by-Demand-in-San-Francisco.pdf).

The conditional commercial-composition estimates and resident mode shares describe spatial context, not identified demand shifts. Our data therefore support historically aligned tariff-change research rather than a claim that higher tariffs cause occupancy to rise. The 1(d) elasticity translation is an illustrative benchmark, not a fitted restriction.

### Conclusion

We retain the negative demand hypothesis as theory and interpret the observed conditional associations without treating them as causal effects. Model 3 combines the declared predictor screen with the fit and validation evidence. Economic size, uncertainty and unresolved specification and dependence limits must be read together. The seven-predictor alternative is a sensitivity comparison.


In [7]:
table_dir=root/"reports/tables";table_dir.mkdir(parents=True,exist_ok=True)
_export_chars = (table_dir/"current_final_results.json").write_text(json.dumps({
 "n":len(data),"predictors":predictors,"selection_rule":"all conventional p<.05 slopes in original 12-candidate full OLS",
 "coefficients":json.loads(regression_table.to_json(orient="index")),
 "economic_effects":json.loads(effects.to_json(orient="index")),
 "model_comparison":json.loads(comparison.to_json(orient="index"))},indent=2))
